# Unified Vision-Model Detection Pipeline\n
\n
This notebook implements a complete vision-based detection pipeline and integrates it with the classical tracker.\n

## Section 0: Imports & Config\n

In [ ]:
import sys
import os
import time
import json
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Import existing modules
sys.path.insert(0, '.')
from data_io import list_samples, load_geff, sample_geff_path, sample_zarr_path, load_volume_frame, VOXEL_SCALE
from detector import detect_cells_threshold
from tracker import build_tracks, detect_divisions
from metric import evaluate_sample, evaluate_all

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
DATA_DIR = "biohub-cell-tracking-during-development"
SPLIT = "train"

# Data subset for this run
NUM_TRAIN_SAMPLES = 150
NUM_EVAL_SAMPLES = 49
FRAMES_PER_SAMPLE = 100  # Full frames

# Patch Geometry & Hyperparams
CROP_YX = 64
POOL_Z = 4
GAUSSIAN_SIGMA = 2.0
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3

# Tracker Config
MAX_LINK_DISTANCE_UM = 7.0

## Section 1: Unified Dataset Construction\n

In [ ]:
def build_node_table(data_dir, split, sample_names):
    rows = []
    for sample in sample_names:
        geff_path = sample_geff_path(data_dir, split, sample)
        geff = load_geff(geff_path)
        df = geff['nodes_df']
        edges = geff['edges']
        
        # Track IDs (connected components)
        G = nx.Graph()
        G.add_nodes_from(df['node_id'])
        if len(edges) > 0:
            G.add_edges_from(edges)
        cc = list(nx.connected_components(G))
        node_to_track = {}
        for track_id, comp in enumerate(cc):
            for n in comp:
                node_to_track[n] = track_id
                
        # Divisions
        if len(edges) > 0:
            outgoing_counts = Counter(edges[:, 0].tolist())
        else:
            outgoing_counts = {}
            
        for _, row in df.iterrows():
            nid = int(row['node_id'])
            rows.append({
                'dataset': sample,
                'node_id': nid,
                't': int(row['t']),
                'z': int(row['z']),
                'y': int(row['y']),
                'x': int(row['x']),
                'track_id': node_to_track.get(nid, -1),
                'is_division_parent': outgoing_counts.get(nid, 0) >= 2
            })
    return pd.DataFrame(rows)

def extract_patch(volume_frame, z, y, x, crop_yx=CROP_YX, pool_z=POOL_Z, sigma=GAUSSIAN_SIGMA):
    Z, Y, X = volume_frame.shape
    half_y = crop_yx // 2
    half_x = crop_yx // 2
    
    y_min, y_max = int(y) - half_y, int(y) + half_y
    x_min, x_max = int(x) - half_x, int(x) + half_x
    
    # Pad if outside boundaries
    pad_y_before = max(0, -y_min)
    pad_y_after = max(0, y_max - Y)
    pad_x_before = max(0, -x_min)
    pad_x_after = max(0, x_max - X)
    
    y_start, y_end = max(0, y_min), min(Y, y_max)
    x_start, x_end = max(0, x_min), min(X, x_max)
    
    crop = volume_frame[:, y_start:y_end, x_start:x_end]
    if pad_y_before > 0 or pad_y_after > 0 or pad_x_before > 0 or pad_x_after > 0:
        crop = np.pad(crop, ((0,0), (pad_y_before, pad_y_after), (pad_x_before, pad_x_after)), mode='constant')
        
    # Max-pool Z
    Z_crop = crop.shape[0]
    remainder = Z_crop % pool_z
    if remainder != 0:
        crop = np.pad(crop, ((0, pool_z - remainder), (0,0), (0,0)), mode='constant')
    patch = crop.reshape(-1, pool_z, crop_yx, crop_yx).max(axis=1)
    
    # Generate Gaussian target
    loc_z, loc_y, loc_x = z / pool_z, crop_yx / 2.0, crop_yx / 2.0
    target = np.zeros(patch.shape, dtype=np.float32)
    radius = int(sigma * 3)
    pz, py, px = np.ogrid[-radius:radius+1, -radius:radius+1, -radius:radius+1]
    kernel = np.exp(-(px**2 + py**2 + pz**2) / (2 * sigma**2))
    
    cz, cy, cx = int(round(loc_z)), int(round(loc_y)), int(round(loc_x))
    z_min_t, z_max_t = max(0, cz - radius), min(patch.shape[0], cz + radius + 1)
    y_min_t, y_max_t = max(0, cy - radius), min(patch.shape[1], cy + radius + 1)
    x_min_t, x_max_t = max(0, cx - radius), min(patch.shape[2], cx + radius + 1)
    
    kz_min, kz_max = z_min_t - (cz - radius), z_max_t - z_min_t + (z_min_t - (cz - radius))
    ky_min, ky_max = y_min_t - (cy - radius), y_max_t - y_min_t + (y_min_t - (cy - radius))
    kx_min, kx_max = x_min_t - (cx - radius), x_max_t - x_min_t + (x_min_t - (cx - radius))
    
    if kz_min < kz_max and ky_min < ky_max and kx_min < kx_max:
        target[z_min_t:z_max_t, y_min_t:y_max_t, x_min_t:x_max_t] = np.maximum(
            target[z_min_t:z_max_t, y_min_t:y_max_t, x_min_t:x_max_t],
            kernel[kz_min:kz_max, ky_min:ky_max, kx_min:kx_max]
        )
        
    return patch, target, (loc_z, loc_y, loc_x)

class CellPatchDataset(Dataset):
    def __init__(self, data_dir, split, sample_names, frames_per_sample=FRAMES_PER_SAMPLE):
        self.data_dir = data_dir
        self.split = split
        self.df = build_node_table(data_dir, split, sample_names)
        self.df = self.df[self.df['t'] < frames_per_sample].reset_index(drop=True)
        
        # We will dynamically load volumes to save memory
        self.items = []
        for i, row in self.df.iterrows():
            self.items.append({
                'dataset': row['dataset'],
                't': row['t'],
                'z': row['z'], 'y': row['y'], 'x': row['x'],
                'is_positive': True
            })
            
        # Add random negative patches (1 per positive)
        # This solves the class imbalance without hard negatives for now
        np.random.seed(42)
        n_pos = len(self.items)
        for i in range(n_pos):
            row = self.items[i]
            # Random offset at least 32 pixels away from the center cell
            offset_y = np.random.choice(np.concatenate([np.arange(-64, -32), np.arange(32, 64)]))
            offset_x = np.random.choice(np.concatenate([np.arange(-64, -32), np.arange(32, 64)]))
            self.items.append({
                'dataset': row['dataset'],
                't': row['t'],
                'z': row['z'],
                'y': np.clip(row['y'] + offset_y, CROP_YX//2, 256-CROP_YX//2),
                'x': np.clip(row['x'] + offset_x, CROP_YX//2, 256-CROP_YX//2),
                'is_positive': False
            })
            
        print(f"Dataset initialized: {len(self.df)} GT nodes -> {len(self.items)} total patches (50% positive)")
        
    def __len__(self):
        return len(self.items)
        
    def __getitem__(self, idx):
        item = self.items[idx]
        zarr_p = sample_zarr_path(self.data_dir, self.split, item['dataset'])
        vol = load_volume_frame(zarr_p, item['t']).astype(np.float32)
        
        patch, target, _ = extract_patch(vol, item['z'], item['y'], item['x'])
        
        # Normalize patch [0, 1] using robust bounds
        p_low, p_high = np.percentile(patch, [1, 99.9])
        patch = np.clip((patch - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
        if not item['is_positive']:
            # For negatives, the patch might still contain another cell if we were unlucky,
            # but for simplicity we assume the target is all zeros for now.
            target = np.zeros_like(target)
            
        patch_t = torch.from_numpy(patch).unsqueeze(0)
        target_t = torch.from_numpy(target).unsqueeze(0)
        return patch_t, target_t

In [ ]:
# Validate Schema and Dataset
all_samples = list_samples(DATA_DIR, SPLIT)
train_samples = all_samples[:NUM_TRAIN_SAMPLES]
eval_samples = all_samples[NUM_TRAIN_SAMPLES:NUM_TRAIN_SAMPLES+NUM_EVAL_SAMPLES]

print("Building train dataset...")
train_dataset = CellPatchDataset(DATA_DIR, SPLIT, train_samples)
print("\nFirst 5 rows of node table:")
display(train_dataset.df.head())

# Show a positive and negative patch
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
pos_patch, pos_target = train_dataset[0]
neg_patch, neg_target = train_dataset[len(train_dataset)//2] # negative counterpart

axes[0,0].imshow(pos_patch.squeeze().max(dim=0).values, cmap='gray')
axes[0,0].set_title('Positive Patch (MIP)')
axes[0,1].imshow(pos_target.squeeze().max(dim=0).values, cmap='hot')
axes[0,1].set_title('Positive Target')

axes[1,0].imshow(neg_patch.squeeze().max(dim=0).values, cmap='gray')
axes[1,0].set_title('Negative Patch (MIP)')
axes[1,1].imshow(neg_target.squeeze().max(dim=0).values, cmap='hot')
axes[1,1].set_title('Negative Target')

plt.tight_layout()
plt.show()

## Section 2: Vision Model + Training\n

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class UNet3DSmall(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[16, 32, 64]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose3d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv3d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            
            if x.shape != skip_connection.shape:
                x = F.interpolate(x, size=skip_connection.shape[2:], mode="trilinear", align_corners=False)
                
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](concat_skip)

        return torch.sigmoid(self.final_conv(x))

In [ ]:
# Initialize model and dataloader
model = UNet3DSmall().to(DEVICE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def weighted_mse_loss(pred, target, pos_weight=10.0):
    weight = torch.ones_like(target)
    weight[target > 0.05] = pos_weight
    loss = (pred - target) ** 2
    return (loss * weight).mean()

print("Starting training...")
model.train()
for epoch in range(EPOCHS):
    t0 = time.time()
    epoch_loss = 0.0
    for vols, targets in train_loader:
        vols, targets = vols.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad()
        preds = model(vols)
        loss = weighted_mse_loss(preds, targets)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss/len(train_loader):.4f} Time: {time.time()-t0:.1f}s")
print("Training complete.")

In [ ]:
def detect_cells_unet(volume_3d, model, pool_z=POOL_Z, patch_size=CROP_YX, overlap=32, threshold=0.1, min_dist=4):
    """Run inference using sliding window over the full frame."""
    model.eval()
    
    Z, Y, X = volume_3d.shape
    remainder = Z % pool_z
    if remainder != 0:
        vol_pad = np.pad(volume_3d, ((0, pool_z - remainder), (0,0), (0,0)), mode='constant')
    else:
        vol_pad = volume_3d
    
    vol_pooled = vol_pad.reshape(-1, pool_z, Y, X).max(axis=1).astype(np.float32)
    p_low, p_high = np.percentile(vol_pooled, [1, 99.9])
    vol_norm = np.clip((vol_pooled - p_low) / (p_high - p_low + 1e-8), 0, 1)
    
    Z_p, Y_p, X_p = vol_norm.shape
    stride = patch_size - overlap
    heatmap_accum = np.zeros_like(vol_norm)
    
    with torch.no_grad():
        for y in range(0, Y_p - patch_size + stride, stride):
            for x in range(0, X_p - patch_size + stride, stride):
                y_s = min(y, Y_p - patch_size)
                x_s = min(x, X_p - patch_size)
                
                patch = vol_norm[:, y_s:y_s+patch_size, x_s:x_s+patch_size]
                patch_t = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
                
                pred = model(patch_t).squeeze().cpu().numpy()
                
                heatmap_accum[:, y_s:y_s+patch_size, x_s:x_s+patch_size] = np.maximum(
                    heatmap_accum[:, y_s:y_s+patch_size, x_s:x_s+patch_size], pred
                )
                
    centroids_pooled = peak_local_max(heatmap_accum, min_distance=min_dist, threshold_abs=threshold)
    
    if len(centroids_pooled) == 0:
        return np.empty((0, 3), dtype=np.float64), heatmap_accum
        
    centroids_orig = centroids_pooled.astype(np.float64)
    centroids_orig[:, 0] = centroids_orig[:, 0] * pool_z + (pool_z / 2.0)
    
    return centroids_orig, heatmap_accum

In [ ]:
# Evaluate on a held-out frame
eval_sample = eval_samples[0]
eval_zarr_p = sample_zarr_path(DATA_DIR, SPLIT, eval_sample)
eval_frame = load_volume_frame(eval_zarr_p, 10)

geff = load_geff(sample_geff_path(DATA_DIR, SPLIT, eval_sample))
df_eval = geff['nodes_df']
gt_nodes = df_eval[df_eval['t'] == 10]
print(f"GT cells in frame: {len(gt_nodes)}")

t0 = time.time()
preds, pred_heatmap = detect_cells_unet(eval_frame, model)
print(f"Predicted cells in frame: {len(preds)}, time: {time.time()-t0:.2f}s")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(eval_frame.max(axis=0), cmap='gray')
if len(gt_nodes) > 0:
    axes[0].scatter(gt_nodes['x'], gt_nodes['y'], c='lime', marker='o', label='GT')
axes[0].set_title("Input MIP + GT")
axes[0].legend()

axes[1].imshow(pred_heatmap.max(axis=0), cmap='hot')
if len(preds) > 0:
    axes[1].scatter(preds[:, 2], preds[:, 1], c='cyan', marker='x', label='Pred')
axes[1].set_title("Predicted Heatmap MIP + Pred Peaks")
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 3: Cross-Verification with Classical Tracker\n

In [ ]:
# Stage 3: Cross-Verification with Classical Tracker
# We replace the classical detector with our U-Net, but keep the exact same tracking logic.

def run_unet_pipeline(zarr_path, model, frames=FRAMES_PER_SAMPLE):
    all_centroids = {}
    print(f"Detecting cells across {frames} frames...")
    
    t0 = time.time()
    for t in range(frames):
        frame = load_volume_frame(zarr_path, t)
        centroids, _ = detect_cells_unet(frame, model)
        all_centroids[t] = centroids
        if (t+1) % 5 == 0:
            print(f"  Frame {t+1}/{frames} complete.")
            
    print(f"Detection finished in {time.time()-t0:.1f}s. Building tracks...")
    
    t1 = time.time()
    graph = build_tracks(all_centroids, max_distance=MAX_LINK_DISTANCE_UM, voxel_scale=VOXEL_SCALE)
    graph = detect_divisions(graph, all_centroids)
    print(f"Tracking finished in {time.time()-t1:.1f}s.")
    
    return graph, all_centroids

# Validate on 1 sample (first 10 frames)
graph_unet, centroids_unet = run_unet_pipeline(eval_zarr_p, model, frames=10)
print(f"U-Net Graph: {len(graph_unet.nodes)} nodes, {len(graph_unet.edges)} edges.")


## Section 4: Evaluation\n

In [ ]:
# Stage 4: Comparative Evaluation
# We evaluate the U-Net pipeline against the Classical pipeline on our evaluation samples.

results = []

for sample in eval_samples:
    print(f"--- Evaluating {sample} ---")
    zarr_p = sample_zarr_path(DATA_DIR, SPLIT, sample)
    geff_p = sample_geff_path(DATA_DIR, SPLIT, sample)
    
    print("Running U-Net pipeline...")
    graph_unet, _ = run_unet_pipeline(zarr_p, model, frames=10)
    
    print("Running Classical pipeline...")
    all_centroids_cl = {}
    for t in range(10):
        frame = load_volume_frame(zarr_p, t)
        all_centroids_cl[t] = detect_cells_threshold(frame, VOXEL_SCALE)
    graph_cl = build_tracks(all_centroids_cl, max_distance=MAX_LINK_DISTANCE_UM, voxel_scale=VOXEL_SCALE)
    graph_cl = detect_divisions(graph_cl, all_centroids_cl)
    
    print("Scoring...")
    metrics_unet = evaluate_sample(geff_p, graph_unet)
    metrics_cl = evaluate_sample(geff_p, graph_cl)
    
    results.append({
        'Sample': sample,
        'Method': 'Classical',
        'Edge Jaccard': metrics_cl['edge_jaccard'],
        'Div Jaccard': metrics_cl['division_jaccard']
    })
    results.append({
        'Sample': sample,
        'Method': 'U-Net',
        'Edge Jaccard': metrics_unet['edge_jaccard'],
        'Div Jaccard': metrics_unet['division_jaccard']
    })

df_results = pd.DataFrame(results)
display(df_results)
